# 04 — send one confirmed-defect notification

The only thing in this project that sends mail, and it does one job: take a
message the app has already composed and put it on an SMTP server.

**It composes nothing and decides nothing.** The app decided — `reviewed` with
`decision = 'accepted'`, by a platform-vouched human, read back out of
`results.disposition` before this job was triggered. The wording is
`dq_app/domain/notify.py`. If you want to change what the email says, change it
there, where it is unit-tested, not here.

**Why this job exists rather than an `smtplib` call inside the app.** Credentials.
The app's service principal needs `CAN_RUN` on this job and nothing else: no SMTP
password, no provider key, no secret resource on the app at all. The mail
credential lives here, with the only thing that uses it.

## Set up

```
databricks secrets create-scope dq-notify
databricks secrets put-secret dq-notify smtp_host
databricks secrets put-secret dq-notify smtp_user
databricks secrets put-secret dq-notify smtp_password
databricks secrets put-secret dq-notify smtp_from
```

Then create a job with this notebook as its one task, grant the app's service
principal `CAN_RUN`, and put the job id in `DQ_NOTIFY_JOB_ID`.

Outbound egress to the SMTP host must be permitted. It is by default; a workspace
with an Enterprise-tier network policy needs the host allowlisted, and a refusal
shows up in `system.access.outbound_network`.

In [ ]:
dbutils.widgets.text("recipients", "")
dbutils.widgets.text("subject", "")
dbutils.widgets.text("body_text", "")
dbutils.widgets.text("body_html", "")
dbutils.widgets.text("cohort_id", "")
dbutils.widgets.text("disposition_id", "")

SCOPE = "dq-notify"

recipients = [a.strip() for a in dbutils.widgets.get("recipients").split(",") if a.strip()]
subject = dbutils.widgets.get("subject")
body_text = dbutils.widgets.get("body_text")
body_html = dbutils.widgets.get("body_html")
disposition_id = dbutils.widgets.get("disposition_id")

# Fail loudly and before connecting to anything. A run triggered with no
# recipients is a wiring bug in the caller, and a job that quietly succeeds on it
# is a notification path that reports healthy while delivering nothing.
missing = [n for n, v in [("recipients", recipients), ("subject", subject),
                          ("body_text", body_text)] if not v]
if missing:
    raise ValueError(f"nothing to send - missing: {', '.join(missing)}")

print(f"{len(recipients)} recipient(s) - disposition {disposition_id}")

In [ ]:
import smtplib
from email.message import EmailMessage

host = dbutils.secrets.get(SCOPE, "smtp_host")
user = dbutils.secrets.get(SCOPE, "smtp_user")
password = dbutils.secrets.get(SCOPE, "smtp_password")
sender = dbutils.secrets.get(SCOPE, "smtp_from")
keys = [s.key for s in dbutils.secrets.list(SCOPE)]
port = int(dbutils.secrets.get(SCOPE, "smtp_port")) if "smtp_port" in keys else 587

msg = EmailMessage()
msg["Subject"] = subject
msg["From"] = sender
msg["To"] = ", ".join(recipients)
msg.set_content(body_text)
if body_html:
    msg.add_alternative(body_html, subtype="html")

# The From is this mailbox, never the steward - they did not send it, they made a
# decision that caused it. Their name and the reason they typed are in the body.
with smtplib.SMTP(host, port, timeout=30) as smtp:
    smtp.starttls()
    smtp.login(user, password)
    smtp.send_message(msg)

print(f"sent - {subject}")

## What this job does not do

**It does not record that it sent.** `results.disposition` is append-only and the
app's two writes are the whole write surface; a `notified` event would be a third,
and a third write is a grant and a decision, not a notebook change. What exists
instead is this job's own run history.

**It does not retry.** A failed run is a failed run, visible in that history. The
register row survives regardless - which is the point of the ordering: the
steward's decision is recorded whether or not anyone was successfully told.

**Nothing reads it back.** The backstop for a send that failed silently is a
scheduled query over `results.disposition` on a wide window - the SQL-alert design
this job is the immediate-delivery alternative to. The two are complements: run
the alert daily and it finds any accepted review that never produced a run here.